# W1-11 / EEC-15: MFCC + SVM pilot sanity baseline
CPU is enough (no GPU needed). Run the cells top to bottom. If the pilot audio is not
already in your Drive, the notebook downloads **only the 700 pilot clips** (about 0.4 GB)
from the pinned public Hugging Face release and checks every file's SHA256 against the
committed manifests. The script then fits a StandardScaler + RBF SVM on the 500-clip
frozen training pilot only, predicts the 200-clip development pilot, and scores it with
the shared evaluator.

This is a **pilot sanity reference**, not a competitive baseline and not an
out-of-domain result. Outputs are written to a new folder on Drive; nothing is overwritten.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
# Your own Drive folder. It is created if missing; teammates with the shared team
# folder can point this at it instead.
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
AUDIO_ROOT = DATA_ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA_ROOT)
print('Audio folder:', AUDIO_ROOT)

In [ ]:
import subprocess, sys, tempfile, json
from datetime import datetime, timezone

# Reviewed implementation with enforced WAV hashes and window validation.
# Pin the executed code independently of the temporary PR branch.
REPOSITORY_URL = 'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git'
PINNED_COMMIT = 'c7d2161ce68aab9c424ce1734d9c42eacd17b9a2'

REPO_ROOT = Path(tempfile.mkdtemp(prefix='mfcc-svm-w1-11-'))
def git(*args):
    subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=True)
git('init', '--quiet')
git('remote', 'add', 'origin', REPOSITORY_URL)
git('fetch', '--quiet', '--depth', '1', 'origin', PINNED_COMMIT)
git('checkout', '--quiet', '--detach', 'FETCH_HEAD')
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if SOURCE_COMMIT != PINNED_COMMIT:
    raise RuntimeError(f'Source mismatch: expected {PINNED_COMMIT}, got {SOURCE_COMMIT}')

try:
    import soundfile  # usually preinstalled on Colab
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile==0.14.0'], check=True)

RUN_ROOT = DATA_ROOT / 'runs/mfcc_svm_w1_11' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
print('Source commit:', SOURCE_COMMIT)
print('Output folder:', RUN_ROOT)

In [ ]:
# Presence check; the experiment verifies every WAV SHA256 before using it.
# Fetch missing WAVs from the pinned release, verifying downloaded bytes too.
def missing_pilot_clips():
    missing = []
    for split in ('train', 'dev'):
        rows = [json.loads(l) for l in (REPO_ROOT / f'data/manifests/{split}_pilot.jsonl').read_text().splitlines() if l.strip()]
        missing += [r['clip_id'] for r in rows if not (AUDIO_ROOT / r['audio_path']).is_file()]
    return missing

missing = missing_pilot_clips()
print(f'{700 - len(missing)} of 700 pilot WAV files already present.')
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements-data.txt')], check=True)
    result = subprocess.run([sys.executable, '-u', str(REPO_ROOT / 'src/data/fetch_pilot_audio.py'),
                             '--audio-root', str(AUDIO_ROOT)], cwd=REPO_ROOT)
    assert result.returncode == 0, 'Some clips could not be fetched or verified; see the messages above.'
    missing = missing_pilot_clips()
assert not missing, f'{len(missing)} pilot WAV files are still missing, e.g. {missing[:3]}'
print('All 700 pilot WAV files present; the experiment will verify every WAV hash.')

In [ ]:
# Train on the training pilot, predict the dev pilot, run the shared evaluator.
# Exit 0 = no clip failures; exit 2 = finished with explicit clip failures (still scored).
command = [sys.executable, '-u', str(REPO_ROOT / 'src/experiments/mfcc_svm_baseline.py'),
           '--audio-root', str(AUDIO_ROOT), '--output-dir', str(RUN_ROOT)]
process = subprocess.Popen(command, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
log_lines = []
for line in process.stdout:
    print(line, end='', flush=True)
    log_lines.append(line)
code_ = process.wait()
if RUN_ROOT.is_dir():
    (RUN_ROOT / 'console.log').write_text(''.join(log_lines), encoding='utf-8')
assert code_ in (0, 2), f'Baseline exited {code_}; see the output above'
print('Exit code:', code_)

In [ ]:
record = json.loads((RUN_ROOT / 'run_record.json').read_text())
metrics = json.loads((RUN_ROOT / 'evaluation/metrics.json').read_text())
record['code']['notebook_source_commit'] = SOURCE_COMMIT
(RUN_ROOT / 'run_record.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n')

print('Model revision:', record['model_revision'])
print('Selected C:', record['model_selection']['selected_C'], '| CV (train only):', record['model_selection']['cv']['results'])
print(f"Dev accuracy: {metrics['accuracy']:.2%} | macro-F1: {metrics['macro_f1']:.4f} | coverage: {metrics['coverage']:.0%}")
print('Counts:', record['counts'])
print('Resources:', record['resources'])
print()
for label, row in metrics['per_dialect'].items():
    print(f"{label:>4}  F1 {row['f1']:.2f}  recall {row['recall']:.2f}  precision {row['precision']:.2f}")

## Hand the result back
The next cell zips the run folder (model, predictions, metrics, confusion matrix,
run record) and downloads it. Attach the zip to the Linear issue / PR so a teammate
can review it. Remember this dev pilot is in-source, so the number is only a sanity reference.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(f'/content/mfcc_svm_w1_11_{RUN_ROOT.name}', 'zip', RUN_ROOT)
print('Saved on Drive:', RUN_ROOT)
files.download(archive)